In [1]:
import pandas as pd
import re
import s3fs
import os
import logging
from typing import Optional

In [2]:
# Configurar logging básico
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Regex precompilados (más eficiente)
URL_PATTERN = re.compile(r"http\S+|www\S+", flags=re.MULTILINE)
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#")
MULTISPACE_PATTERN = re.compile(r"\s+")

def minimal_clean(text: str) -> str:
    text = str(text).lower()
    text = URL_PATTERN.sub("", text)
    text = MENTION_PATTERN.sub("", text)
    text = HASHTAG_PATTERN.sub("", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text

In [3]:
def process_parquet_from_s3(input_prefix: str, output_prefix: str, text_column: str = "text") -> None:
    """Lee particiones Parquet desde S3, limpia el texto y guarda en S3."""
    try:
        fs = s3fs.S3FileSystem()
        files = fs.ls(input_prefix)
        
        if not files:
            print(f"No se encontraron archivos en {input_prefix}")
            return

        for file in files:
            print(f"Procesando {file}...")
            
            # Leer partición
            df = pd.read_parquet(f"s3://{file}", filesystem=fs)
            
            # Aplicar limpieza mínima
            df["clean_text"] = df[text_column].apply(minimal_clean)
            
            # Definir ruta de salida
            filename = os.path.basename(file)
            output_path = f"{output_prefix}/{filename}"
            
            # Guardar partición procesada
            df.to_parquet(
                f"s3://{output_path}",
                engine="pyarrow",
                index=False,
                compression="snappy",
                filesystem=fs
            )
            print(f"Guardado exitosamente en s3://{output_path}")

        print(f"Procesamiento completado para {input_prefix}")
        
    except Exception as e:
        print(f"Error procesando datos: {str(e)}")

In [4]:
process_parquet_from_s3(
    input_prefix="parcial-pln/data/raw/train_parquet",
    output_prefix="parcial-pln/data/processed/limpieza_minima/train_parquet"
)

2026-03-04 03:00:59,489 - INFO - Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


Procesando parcial-pln/data/raw/train_parquet/part_0.parquet...
Guardado exitosamente en s3://parcial-pln/data/processed/limpieza_minima/train_parquet/part_0.parquet
Procesando parcial-pln/data/raw/train_parquet/part_1.parquet...
Guardado exitosamente en s3://parcial-pln/data/processed/limpieza_minima/train_parquet/part_1.parquet
Procesando parcial-pln/data/raw/train_parquet/part_10.parquet...
Guardado exitosamente en s3://parcial-pln/data/processed/limpieza_minima/train_parquet/part_10.parquet
Procesando parcial-pln/data/raw/train_parquet/part_11.parquet...
Guardado exitosamente en s3://parcial-pln/data/processed/limpieza_minima/train_parquet/part_11.parquet
Procesando parcial-pln/data/raw/train_parquet/part_12.parquet...
Guardado exitosamente en s3://parcial-pln/data/processed/limpieza_minima/train_parquet/part_12.parquet
Procesando parcial-pln/data/raw/train_parquet/part_13.parquet...
Guardado exitosamente en s3://parcial-pln/data/processed/limpieza_minima/train_parquet/part_13.parq